# Extracción de Sumillas - Universidad del Pacífico (UP)
Extrae los cursos del Departamento Académico de Ingeniería (páginas 83 a 102) de `Sumillas-de-asignaturas-obligatorias_Plan-de-Estudios-2022.pdf`.

In [1]:
import re
import subprocess
import pandas as pd

In [2]:
pdf = "Sumillas-de-asignaturas-obligatorias_Plan-de-Estudios-2022.pdf"

# Extrae todos los cursos del PDF
texto = subprocess.check_output(["pdftotext", pdf, "-"], text=True)
lineas = [
    l.strip() for l in texto.splitlines()
    if l.strip() and l.strip() != "UNIVERSIDAD DEL PACÍFICO" and not l.strip().isdigit()
]

filas = []
curso_actual = None
sumilla = []
esperando_nombre = False

for l in lineas:
    if l == "CURSO":
        if curso_actual:
            filas.append({"curso": curso_actual, "sumilla": " ".join(sumilla)})
        esperando_nombre = True
        sumilla = []
    elif esperando_nombre:
        curso_actual = l
        esperando_nombre = False
    elif curso_actual:
        sumilla.append(l)

if curso_actual:
    filas.append({"curso": curso_actual, "sumilla": " ".join(sumilla)})

df_up = pd.DataFrame(filas)
print(f"Total cursos extraídos: {len(df_up)}")
df_up.head()

Total cursos extraídos: 225


,curso,sumilla
0,Análisis Multivariado para los Negocios,Número de créditos: 4 Código: 141036 Requisito...
1,Analítica de Datos para los Negocios,Número de créditos: 3 Código:142277 Requisitos...
2,Business Agility,Número de créditos: 3 Código: 142280 Requisito...
3,Creación de Valor y Toma de Decisiones,Número de créditos: 3 Código: 142077 Requisito...
4,Dirección Estratégica,Número de créditos: 4 Código: 149078 Requisito...


## Emparejamiento de habilidades por curso (fuerza bruta)

In [3]:
HABILIDADES = {
    # Programación y Datos
    "POO": ["poo", "programación orientada a objetos", "orientada a objetos"],
    "SQL / Bases de Datos": ["sql", "base de datos", "bases de datos", "mysql", "postgresql", "oracle", "modelamiento de datos"],
    "NoSQL": ["nosql", "mongodb"],
    "Python": ["python"],
    "Java": ["java"],
    "C/C++": ["c++", "lenguaje c"],
    "JavaScript / Web": ["javascript", "html", "css", "desarrollo web", "aplicaciones web"],
    "Desarrollo Móvil": ["móvil", "móviles", "android"],
    "Algoritmos y Estructuras": ["algoritmos", "estructuras de datos"],
    "Git": ["git", "control de versiones"],
    "Linux": ["linux", "unix"],
    "Sistemas Operativos": ["sistemas operativos", "sistema operativo"],
    "Cloud Computing": ["cloud", "nube", "aws", "azure"],
    "Big Data / Minería de Datos": ["big data", "minería de datos", "data mining", "ciencia de datos", "analítica de datos"],
    "Machine Learning / IA": ["machine learning", "aprendizaje automático", "inteligencia artificial", "redes neuronales"],
    "Ciberseguridad": ["ciberseguridad", "seguridad informática", "seguridad de la información", "criptografía"],
    "Redes": ["redes", "tcp/ip", "cisco", "enrutamiento", "conmutación"],
    # Electrónica, Control y Mecatrónica
    "MATLAB / Simulink": ["matlab", "simulink"],
    "AutoCAD / CAD": ["autocad", "cad", "solidworks", "diseño asistido"],
    "PLC / Automatización": ["plc", "autómatas programables", "automatización", "scada"],
    "Arduino": ["arduino"],
    "Sistemas Embebidos": ["embebidos", "microcontroladores", "microprocesadores"],
    "FPGA / VHDL": ["fpga", "vhdl"],
    "Control Automático": ["control automático", "sistemas de control", "pid"],
    "Sensores e Instrumentación": ["sensores", "actuadores", "instrumentación"],
    "Neumática e Hidráulica": ["neumática", "hidráulica", "electroneumática"],
    "Electrónica": ["electrónica", "circuitos"],
    "Telecomunicaciones": ["telecomunicaciones", "señales", "antenas", "fibra óptica"],
    "IoT": ["iot", "internet de las cosas"],
    "Robótica": ["robótica", "robots"],
    # Industrial, Gestión y Analítica
    "Excel": ["excel"],
    "Power BI": ["power bi", "inteligencia de negocios", "business intelligence"],
    "ERP / SAP": ["erp", "sap"],
    "Estadística": ["estadística", "probabilidades"],
    "Investigación de Operaciones": ["investigación de operaciones", "programación lineal", "optimización"],
    "Lean / Six Sigma": ["lean", "six sigma", "mejora continua"],
    "Gestión de Calidad": ["gestión de calidad", "calidad total", "iso 9001"],
    "Logística / Supply Chain": ["logística", "cadena de suministro", "supply chain", "inventarios"],
    "Gestión de Proyectos": ["gestión de proyectos", "pmbok", "scrum", "ágil", "ágiles"],
    "Seguridad Industrial": ["seguridad industrial", "salud ocupacional", "sst"],
    "Costos y Finanzas": ["costos", "presupuestos", "evaluación de proyectos", "finanzas"],
    # Transversales e Idiomas
    "Inglés": ["inglés"],
    "Trabajo en Equipo": ["trabajo en equipo"],
    "Liderazgo": ["liderazgo"],
    "Comunicación Efectiva": ["comunicación efectiva", "comunicación oral y escrita"],
}


def buscar_habilidades(texto):
    texto = str(texto).lower()
    encontradas = []
    for hab, variantes in HABILIDADES.items():
        for v in variantes:
            if re.search(rf"\b{re.escape(v)}\b", texto):
                encontradas.append(hab)
                break
    return encontradas


df_up["habilidades"] = df_up["sumilla"].apply(buscar_habilidades)
df_up[["curso", "sumilla", "habilidades"]].head(10)

,curso,sumilla,habilidades
0,Análisis Multivariado para los Negocios,Número de créditos: 4 Código: 141036 Requisito...,"[Big Data / Minería de Datos, Logística / Supp..."
1,Analítica de Datos para los Negocios,Número de créditos: 3 Código:142277 Requisitos...,"[SQL / Bases de Datos, Python, Excel, Power BI..."
2,Business Agility,Número de créditos: 3 Código: 142280 Requisito...,"[Lean / Six Sigma, Gestión de Proyectos, Lider..."
3,Creación de Valor y Toma de Decisiones,Número de créditos: 3 Código: 142077 Requisito...,"[Lean / Six Sigma, Gestión de Proyectos, Costo..."
4,Dirección Estratégica,Número de créditos: 4 Código: 149078 Requisito...,"[Logística / Supply Chain, Costos y Finanzas, ..."
5,Diseño Organizacional y Estrategia,Número de créditos: 4 Código: 142081 Requisito...,"[Liderazgo, Comunicación Efectiva]"
6,Emprendimiento e Innovación,Número de créditos: 4 Código: 142082 Requisito...,"[Big Data / Minería de Datos, Machine Learning..."
7,Estrategia,Número de créditos: 4 Código: 142083 Requisito...,"[Costos y Finanzas, Liderazgo, Comunicación Ef..."
8,Fundamentos de las Ciencias Empresariales,Número de créditos: 4 Código: 141038 Requisito...,"[Costos y Finanzas, Liderazgo, Comunicación Ef..."
9,Gestión de la Cadena de Suministros,Número de créditos: 4 Código: 141041 Requisito...,"[Logística / Supply Chain, Liderazgo]"


In [4]:
df_up.to_csv("up_sumillas_habilidades.csv", index=False)
print("Guardado en up_sumillas_habilidades.csv")

Guardado en up_sumillas_habilidades.csv
